In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import regexp_extract

dataset_judicial = spark.read.format("delta").load(
    "/Volumes/workspace/default/tfm_visnu_raw/dataset_judicial_inicial_delta"
)

display(dataset_judicial)

anio,orgjud,denuncias,victimas,ordenes_proteccion,quebrantamientos
2014,J1I NR. 1 ORIHUELA,0.0,0.0,0.0,0.0
2014,J1I NR. 3 CACERES,247.0,247.0,68.0,24.0
2014,J1I NR. 4 BENIDORM,0.0,0.0,0.0,0.0
2014,J1I NR. 4 DENIA,0.0,0.0,0.0,0.0
2014,J1I NR. 4 TORRENT,0.0,0.0,0.0,0.0
2014,J1II NR. 1 A ESTRADA,39.0,39.0,6.0,0.0
2014,J1II NR. 1 AGUILAR,35.0,35.0,9.0,1.0
2014,J1II NR. 1 ALCALA LA REAL,59.0,59.0,9.0,0.0
2014,J1II NR. 1 ALCARAZ,14.0,14.0,0.0,1.0
2014,J1II NR. 1 ALCAZAR DE SAN JUAN,79.0,79.0,44.0,4.0


In [0]:
dataset_con_ciudad = dataset_judicial.withColumn(
    "ciudad",
    regexp_extract("orgjud", r"NR\.\s*\d+\s+(.*)", 1)
)

display(dataset_con_ciudad)

anio,orgjud,denuncias,victimas,ordenes_proteccion,quebrantamientos,ciudad
2014,J1I NR. 1 ORIHUELA,0.0,0.0,0.0,0.0,ORIHUELA
2014,J1I NR. 3 CACERES,247.0,247.0,68.0,24.0,CACERES
2014,J1I NR. 4 BENIDORM,0.0,0.0,0.0,0.0,BENIDORM
2014,J1I NR. 4 DENIA,0.0,0.0,0.0,0.0,DENIA
2014,J1I NR. 4 TORRENT,0.0,0.0,0.0,0.0,TORRENT
2014,J1II NR. 1 A ESTRADA,39.0,39.0,6.0,0.0,A ESTRADA
2014,J1II NR. 1 AGUILAR,35.0,35.0,9.0,1.0,AGUILAR
2014,J1II NR. 1 ALCALA LA REAL,59.0,59.0,9.0,0.0,ALCALA LA REAL
2014,J1II NR. 1 ALCARAZ,14.0,14.0,0.0,1.0,ALCARAZ
2014,J1II NR. 1 ALCAZAR DE SAN JUAN,79.0,79.0,44.0,4.0,ALCAZAR DE SAN JUAN


In [0]:
dataset_con_ciudad_norm = (
    dataset_con_ciudad
    .withColumn("ciudad", F.upper(F.trim(F.col("ciudad"))))
    .withColumn("ciudad", F.regexp_replace("ciudad", r"\s+", " "))
)

In [0]:
mapping_final = [
    ("A CORUÑA", "A CORUÑA"),
    ("A ESTRADA", "PONTEVEDRA"),
    ("AGUILAR", "CORDOBA"),
    ("ALBACETE", "ALBACETE"),
    ("ALCALA DE GUADAIRA", "SEVILLA"),
    ("ALCALA DE HENARES", "MADRID"),
    ("ALCALA LA REAL", "JAEN"),
    ("ALCARAZ", "ALBACETE"),
    ("ALCAZAR DE SAN JUAN", "CIUDAD REAL"),
    ("ALCAÑIZ", "TERUEL"),
    ("ALCOBENDAS", "MADRID"),
    ("ALCORCON", "MADRID"),
    ("ALCOY", "ALICANTE"),
    ("ALGECIRAS", "CADIZ"),
    ("ALICANTE/ALACANT", "ALICANTE"),
    ("ALMADEN", "CIUDAD REAL"),
    ("ALMAGRO", "CIUDAD REAL"),
    ("ALMANSA", "ALBACETE"),
    ("ALMAZAN", "SORIA"),
    ("ALMENDRALEJO", "BADAJOZ"),
    ("ALMERIA", "ALMERIA"),
    ("ANTEQUERA", "MALAGA"),
    ("ARACENA", "HUELVA"),
    ("ARANDA DE DUERO", "BURGOS"),
    ("ARCOS DE LA FRONTERA", "CADIZ"),
    ("BADALONA", "BARCELONA"),
    ("BAZA", "GRANADA"),
    ("BERJA", "ALMERIA"),
    ("COLLADO VILLALBA", "MADRID"),
    ("CORCUBION", "A CORUÑA"),
    ("CUENCA", "CUENCA"),
    ("DURANGO", "BIZKAIA"),
    ("FONSAGRADA", "LUGO"),
    ("FREGENAL DE LA SIERRA", "BADAJOZ"),
    ("GETAFE", "MADRID"),
    ("GUADIX", "GRANADA"),
    ("HUESCAR", "GRANADA"),
    ("JACA", "HUESCA"),
    ("JEREZ DE LA FRONTERA", "CADIZ"),
    ("LAREDO", "CANTABRIA"),
    ("LEGANES", "MADRID"),
    ("LORCA", "MURCIA"),
    ("MADRID", "MADRID"),
    ("MALAGA", "MALAGA"),
    ("MEDINA DEL CAMPO", "VALLADOLID"),
    ("MERIDA", "BADAJOZ"),
    ("MOTILLA DEL PALANCAR", "CUENCA"),
    ("MURCIA", "MURCIA"),
    ("NAVALMORAL DE LA MATA", "CACERES"),
    ("NEGREIRA", "A CORUÑA"),
    ("NOIA", "A CORUÑA"),
    ("OCAÑA", "TOLEDO"),
    ("OLIVENZA", "BADAJOZ"),
    ("OLOT", "GIRONA"),
    ("ONTINYENT", "VALENCIA"),
    ("ORIHUELA", "ALICANTE"),
    ("REUS", "TARRAGONA"),
    ("SAN BARTOLOME DE TIRAJANA", "LAS PALMAS"),
    ("SAN VICENTE DEL RASPEIG", "ALICANTE"),
    ("SANTA CRUZ DE TENERIFE", "SANTA CRUZ DE TENERIFE"),
    ("SIGÜENZA", "GUADALAJARA"),
    ("TORTOSA", "TARRAGONA")
]

In [0]:
df_mapping = spark.createDataFrame(mapping_final, ["ciudad", "provincia"])
display(df_mapping)

ciudad,provincia
A CORUÑA,A CORUÑA
A ESTRADA,PONTEVEDRA
AGUILAR,CORDOBA
ALBACETE,ALBACETE
ALCALA DE GUADAIRA,SEVILLA
ALCALA DE HENARES,MADRID
ALCALA LA REAL,JAEN
ALCARAZ,ALBACETE
ALCAZAR DE SAN JUAN,CIUDAD REAL
ALCAÑIZ,TERUEL


In [0]:
dataset_prov = dataset_con_ciudad_norm.join(df_mapping, on="ciudad", how="left")

In [0]:
ciudades_pendientes = (
    dataset_prov
    .filter(F.col("provincia").isNull())
    .select("ciudad")
    .distinct()
    .orderBy("ciudad")
)

display(ciudades_pendientes)

ciudad
""
ALMUÑECAR
ALZIRA
AMPOSTA
AMURRIO
ANDUJAR
AOIZ
ARANJUEZ
ARCHIDONA
ARENAS DE SAN PEDRO


In [0]:
display(
    ciudades_pendientes.selectExpr("count(*) as num_ciudades_pendientes")
)

num_ciudades_pendientes
372


In [0]:
# dataset_judicial_provincia = (
#     dataset_prov
#     .filter(F.col("provincia").isNotNull())
#     .groupBy("anio", "provincia")
#     .agg(
#         F.sum("denuncias").alias("denuncias"),
#         F.sum("victimas").alias("victimas"),
#         F.sum("ordenes_proteccion").alias("ordenes_proteccion"),
#         F.sum("quebrantamientos").alias("quebrantamientos")
#     )
#     .orderBy("anio", "provincia")
# )

# display(dataset_judicial_provincia)

In [0]:
# dataset_judicial_provincia.write.mode("overwrite").format("delta").save(
#     "/Volumes/workspace/default/tfm_visnu_raw/dataset_judicial_provincia_delta"
# )

In [0]:
import pandas as pd

ruta_ine = "/Volumes/workspace/default/tfm_visnu_raw/ine_municipios_provincias.xlsx"

xls = pd.ExcelFile(ruta_ine)
print(xls.sheet_names[:10], "...")
print("Número de hojas:", len(xls.sheet_names))

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10'] ...
Número de hojas: 52


In [0]:
dfs_ine = []

for hoja in xls.sheet_names:
    # Leer sin cabecera para capturar provincia y luego reasignar nombres
    temp = pd.read_excel(ruta_ine, sheet_name=hoja, header=None)

    # Provincia está en fila 1, columna 0
    provincia = temp.iloc[1, 0]

    # Encabezados reales están en fila 2
    datos = temp.iloc[2:].copy()
    datos.columns = ["cpro", "cmun", "dc", "ciudad"]
    datos["provincia"] = provincia

    dfs_ine.append(datos)

df_ine_pd = pd.concat(dfs_ine, ignore_index=True)

print(df_ine_pd.shape)
df_ine_pd.head()

(8184, 5)


,cpro,cmun,dc,ciudad,provincia
0,CPRO,CMUN,DC,NOMBRE,Araba/Álava
1,01,051,3,Agurain/Salvatierra,Araba/Álava
2,01,001,4,Alegría-Dulantzi,Araba/Álava
3,01,002,9,Amurrio,Araba/Álava
4,01,049,3,Añana,Araba/Álava


In [0]:
df_ine_pd = df_ine_pd.dropna(subset=["ciudad"]).copy()

# Quitar posibles filas basura repetidas
df_ine_pd = df_ine_pd[df_ine_pd["ciudad"] != "NOMBRE"]

# Pasar todo a string
for col in ["cpro", "cmun", "dc", "ciudad", "provincia"]:
    df_ine_pd[col] = df_ine_pd[col].astype(str).str.strip()

print(df_ine_pd.shape)
df_ine_pd.head()

(8132, 5)


,cpro,cmun,dc,ciudad,provincia
1,01,051,3,Agurain/Salvatierra,Araba/Álava
2,01,001,4,Alegría-Dulantzi,Araba/Álava
3,01,002,9,Amurrio,Araba/Álava
4,01,049,3,Añana,Araba/Álava
5,01,003,5,Aramaio,Araba/Álava


In [0]:
df_ine = spark.createDataFrame(df_ine_pd)
display(df_ine)

cpro,cmun,dc,ciudad,provincia
01,051,3,Agurain/Salvatierra,Araba/Álava
01,001,4,Alegría-Dulantzi,Araba/Álava
01,002,9,Amurrio,Araba/Álava
01,049,3,Añana,Araba/Álava
01,003,5,Aramaio,Araba/Álava
01,006,6,Armiñón,Araba/Álava
01,037,6,Arraia-Maeztu,Araba/Álava
01,008,8,Arratzua-Ubarrundia,Araba/Álava
01,004,0,Artziniega,Araba/Álava
01,009,1,Asparrena,Araba/Álava


In [0]:
from pyspark.sql import functions as F

df_ine_norm = (
    df_ine
    .withColumn("ciudad", F.upper(F.trim(F.col("ciudad"))))
    .withColumn("ciudad", F.regexp_replace("ciudad", r"\s+", " "))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Á", "A"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "É", "E"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Í", "I"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ó", "O"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ú", "U"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ü", "U"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ñ", "N"))
    .withColumn("provincia", F.upper(F.trim(F.col("provincia"))))
    .withColumn("provincia", F.regexp_replace("provincia", "Á", "A"))
    .withColumn("provincia", F.regexp_replace("provincia", "É", "E"))
    .withColumn("provincia", F.regexp_replace("provincia", "Í", "I"))
    .withColumn("provincia", F.regexp_replace("provincia", "Ó", "O"))
    .withColumn("provincia", F.regexp_replace("provincia", "Ú", "U"))
    .withColumn("provincia", F.regexp_replace("provincia", "Ü", "U"))
    .withColumn("provincia", F.regexp_replace("provincia", "Ñ", "N"))
    .select("ciudad", "provincia")
    .dropDuplicates(["ciudad"])
)

display(df_ine_norm)

ciudad,provincia
ABABUJ,TERUEL
ABADES,SEGOVIA
ABADIA,CACERES
ABADIN,LUGO
ABADINO,BIZKAIA
ABAIGAR,NAVARRA
ABAJAS,BURGOS
ABALOS,LA RIOJA
ABALTZISKETA,GIPUZKOA
ABANADES,GUADALAJARA


In [0]:
dataset_con_ciudad_norm = (
    dataset_con_ciudad
    .withColumn("ciudad", F.upper(F.trim(F.col("ciudad"))))
    .withColumn("ciudad", F.regexp_replace("ciudad", r"\s+", " "))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Á", "A"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "É", "E"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Í", "I"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ó", "O"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ú", "U"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ü", "U"))
    .withColumn("ciudad", F.regexp_replace("ciudad", "Ñ", "N"))
)

In [0]:
dataset_prov_auto = dataset_con_ciudad_norm.join(df_ine_norm, on="ciudad", how="left")

In [0]:
ciudades_pendientes_auto = (
    dataset_prov_auto
    .filter(F.col("provincia").isNull())
    .select("ciudad")
    .distinct()
    .orderBy("ciudad")
)

display(ciudades_pendientes_auto)

ciudad
""
A CORUNA
A ESTRADA
AGUILAR
ALCOY
ALICANTE/ALACANT
AOIZ
BARCO DE VALDEORRAS
BURGO DE OSMA
CANGAS DE MORRAZO


In [0]:
display(
    ciudades_pendientes_auto.selectExpr("count(*) as num_ciudades_pendientes")
)

num_ciudades_pendientes
60


In [0]:
mapping_manual_residual = [
    ("A CORUNA", "A CORUNA"),
    ("A ESTRADA", "PONTEVEDRA"),
    ("AGUILAR", "CORDOBA"),
    ("ALCOY", "ALICANTE"),
    ("ALICANTE/ALACANT", "ALICANTE"),
    ("AOIZ", "NAVARRA"),
    ("BARCO DE VALDEORRAS", "OURENSE"),
    ("BURGO DE OSMA", "SORIA"),
    ("CANGAS DE MORRAZO", "PONTEVEDRA"),
    ("CANGAS DE NARCEA", "ASTURIAS"),
    ("CASAS IBANEZ", "ALBACETE"),
    ("CASTELLON/CASTELLO", "CASTELLON"),
    ("CERDANYOLA DEL VALLES", "BARCELONA"),
    ("CORNELLA DE LLOBREGAT", "BARCELONA"),
    ("DONOSTIA-SAN SEBASTIAN", "GIPUZKOA"),
    ("EL EJIDO", "ALMERIA"),
    ("EL PRAT DE LLOBREGAT", "BARCELONA"),
    ("EL PUERTO DE SANTA MARIA", "CADIZ"),
    ("EL VENDRELL", "TARRAGONA"),
    ("ELCHE/ELX", "ALICANTE"),
    ("ELX", "ALICANTE"),
    ("ESTELLA", "NAVARRA"),
    ("FONSAGRADA", "LUGO"),
    ("GAVA", "BARCELONA"),
    ("L'HOSPITALET DE LLOBREGAT", "BARCELONA"),
    ("LA ALMUNIA DE DONA GODINA", "ZARAGOZA"),
    ("LA BANEZA", "LEON"),
    ("LA BISBAL D'EMPORDA", "GIRONA"),
    ("LA CAROLINA", "JAEN"),
    ("LA LINEA DE LA CONCEPCION", "CADIZ"),
    ("LA OROTAVA", "SANTA CRUZ DE TENERIFE"),
    ("LA PALMA DEL CONDADO", "HUELVA"),
    ("LA RODA", "ALBACETE"),
    ("LA SEU D'URGELL", "LLEIDA"),
    ("LAS PALMAS DE GRAN CANARIA", "LAS PALMAS"),
    ("LOS LLANOS DE ARIDANE", "SANTA CRUZ DE TENERIFE"),
    ("LUARCA (VALDES)", "ASTURIAS"),
    ("MAO/MAHON", "ILLES BALEARS"),
    ("MOLLET DEL VALLEES", "BARCELONA"),
    ("MONCADA", "VALENCIA"),
    ("O CARBALLINO", "OURENSE"),
    ("O PORRINO", "PONTEVEDRA"),
    ("PAMPLONA", "NAVARRA"),
    ("PAMPLONA/IRUNA", "NAVARRA"),
    ("PILONA-INFIESTO", "ASTURIAS"),
    ("POBRA DE TRIVES", "OURENSE"),
    ("PUENTE-GENIL", "CORDOBA"),
    ("PUIGCERDA", "GIRONA"),
    ("SAGUNTO/SAGUNT", "VALENCIA"),
    ("SAN VICENTE DEL RASPEIG", "ALICANTE"),
    ("VALENCIA", "VALENCIA"),
    ("VELEZ MALAGA", "MALAGA"),
    ("VELEZ RUBIO", "ALMERIA"),
    ("VILAFRANCA DEL PENEDES", "BARCELONA"),
    ("VILLAJOYOSA/LA VILA JOIOSA", "ALICANTE"),
    ("VILLARCAYO", "BURGOS"),
    ("VILLARREAL/VILA-REAL", "CASTELLON"),
    ("VINAROS", "CASTELLON"),
    ("XATIVA", "VALENCIA")
]



In [0]:
df_mapping_manual = spark.createDataFrame(
    mapping_manual_residual,
    ["ciudad", "provincia_manual"]
)

display(df_mapping_manual)

ciudad,provincia_manual
A CORUNA,A CORUNA
A ESTRADA,PONTEVEDRA
AGUILAR,CORDOBA
ALCOY,ALICANTE
ALICANTE/ALACANT,ALICANTE
AOIZ,NAVARRA
BARCO DE VALDEORRAS,OURENSE
BURGO DE OSMA,SORIA
CANGAS DE MORRAZO,PONTEVEDRA
CANGAS DE NARCEA,ASTURIAS


In [0]:
dataset_prov_final_map = (
    dataset_prov_auto
    .join(df_mapping_manual, on="ciudad", how="left")
    .withColumn(
        "provincia_final",
        F.coalesce(F.col("provincia_manual"), F.col("provincia"))
    )
)

In [0]:
ciudades_pendientes_finales = (
    dataset_prov_final_map
    .filter(F.col("provincia_final").isNull())
    .select("ciudad")
    .distinct()
    .orderBy("ciudad")
)

display(ciudades_pendientes_finales)

display(
    ciudades_pendientes_finales.selectExpr("count(*) as num_ciudades_pendientes_finales")
)

ciudad
""
PAMPLONA/IRUÐA


num_ciudades_pendientes_finales
2


In [0]:
display(
    dataset_prov_final_map
    .filter(F.col("provincia_final").isNull())
    .select("ciudad")
    .distinct()
)

ciudad
PAMPLONA/IRUÐA
""


In [0]:
df_mapping_fix = spark.createDataFrame(
    [
        ("PAMPLONA/IRUÐA", "NAVARRA"),
        ("PAMPLONA/IRUÑA", "NAVARRA"),
        ("PAMPLONA/IRUNA", "NAVARRA")
    ],
    ["ciudad", "provincia_fix"]
)

In [0]:
dataset_prov_final_map = (
    dataset_prov_final_map
    .drop("provincia_fix")
    .join(df_mapping_fix, on="ciudad", how="left")
    .withColumn(
        "provincia_final",
        F.coalesce(F.col("provincia_final"), F.col("provincia_fix"))
    )
    .drop("provincia_fix")
)

In [0]:
ciudades_pendientes_finales = (
    dataset_prov_final_map
    .filter(
        (F.col("provincia_final").isNull()) &
        (F.col("ciudad").isNotNull()) &
        (F.trim(F.col("ciudad")) != "")
    )
    .select("ciudad")
    .distinct()
    .orderBy("ciudad")
)

display(ciudades_pendientes_finales)

display(
    ciudades_pendientes_finales.selectExpr("count(*) as num_ciudades_pendientes_finales")
)

ciudad


num_ciudades_pendientes_finales
0


In [0]:
dataset_judicial_provincia = (
    dataset_prov_final_map
    .filter(F.col("provincia_final").isNotNull())
    .groupBy("anio", F.col("provincia_final").alias("provincia"))
    .agg(
        F.sum("denuncias").alias("denuncias"),
        F.sum("victimas").alias("victimas"),
        F.sum("ordenes_proteccion").alias("ordenes_proteccion"),
        F.sum("quebrantamientos").alias("quebrantamientos")
    )
    .orderBy("anio", "provincia")
)

display(dataset_judicial_provincia)

anio,provincia,denuncias,victimas,ordenes_proteccion,quebrantamientos
2014,A CORUNA,1928.0,1928.0,412.0,57.0
2014,ALBACETE,808.0,808.0,332.0,28.0
2014,ALICANTE,3294.0,3294.0,1083.0,70.0
2014,ALICANTE/ALACANT,3061.0,3061.0,708.0,26.0
2014,ALMERIA,2084.0,2084.0,641.0,191.0
2014,ARABA/ALAVA,663.0,663.0,80.0,0.0
2014,ASTURIAS,2394.0,2394.0,741.0,51.0
2014,AVILA,274.0,274.0,122.0,0.0
2014,BADAJOZ,1302.0,1302.0,482.0,17.0
2014,BARCELONA,12383.0,12383.0,3551.0,256.0


In [0]:
display(
    dataset_judicial_provincia
    .select("anio")
    .distinct()
    .orderBy("anio")
)

anio
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023


In [0]:
dataset_judicial_provincia.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/tfm_visnu_raw/dataset_judicial_provincia_delta"
)